In [3]:
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [5]:
tools = [
    {
        "type": "function",
        "name": "get_flight_info",
        "description": "Get flight details between two cities",
        "parameters": {
            "type": "object",
            "properties": {
                "origin": {
                    "type": "string",
                    "description": "Origin city"
                },
                "destination": {
                    "type": "string",
                    "description": "Destination city"
                }
            },
            "required": ["origin", "destination"]
        }
    }
]

In [6]:
#mimic a flight API
from datetime import datetime, timedelta
def get_flight_info(origin, destination):
    """get the flight information between two locations."""
    flight_info = {
        "origin": origin,
        "destination": destination,
        "datetime": str(datetime.now() + timedelta(hours=2)),
        "airline": "Indigo Airlines [IA]",
        "flight": "IA420",
        "available_seats": 20,
    }
    return json.dumps(flight_info)

In [7]:
prompt = "when is the next flight from Ranchi to Port Blair and how many seats are available?"

In [8]:
input_list = [
    {"role": "user", "content": prompt}
]
response3 = client.responses.create(
    model="gpt-5",
    input= input_list,
    tools= tools,
    
)
input_list += response3.output

In [9]:
import json

for item in response3.output:
    if item.type == "function_call" and item.name == "get_flight_info":
        
        args = json.loads(item.arguments)

        flight_data = get_flight_info(
            origin=args["origin"],
            destination=args["destination"]
        )

        # send tool result back to model
        input_list.append({
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": json.dumps(flight_data)
        })

In [10]:
response4 = client.responses.create(
    model="gpt-5",
    tools=tools,
    input=input_list,
    
)

print(response4.output_text)

Here’s the next available flight from Ranchi to Port Blair:

- Airline/Flight: Indigo Airlines (IA420)
- Departure time: 2026-02-27 18:22 (local time)
- Seats available: 20

Would you like me to hold or book seats, or check alternative flights?
